## LLM model
`model = ModelInterface(`
    `model_id,`
    `prams,`
    `credentials,`
    `project_id`
`)`

`model_id = 'meta-llama/llma-302-90b-vision-instruct'`
-- the model is an instruct model, since the name has '-insturct'

`parameters = {GenParams.MAS_NEW_TOKEN: 256,GenParams.TEMPREATURE: 0.2}`

`credentials = {"url": "https://s-south.ml.could.ibm.com"}`

`project_id = "skills-network"`

- to run the model 
    - `model.generate()`
    - `print(msg['result][0]['generated_text'])`


## Chat Model
- force the model to LangChain compatable
    - text-in, text-out model as expected by the LangChain.
    - `WhatsonxLLM(model)`
    - `print(llama_ll.invoke("who is man's best friend?"))`

## Groq

In [4]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
    , temperature=0
    , api_key=GROQ_API_KEY
)

embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

## Google

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

embedding_model = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004'
    , task_type="retrieval_document"
    , google_api_key=gemini_api
)



## Chat messages
- `SystemMessage` 
    - have high weightage, it has presidence over the Human message
    - During RHLF models are thught that the system messages are Master rules
    - Set the presona and the bounderies
- `HumanMessage`
    - task description
- `AIMessage`
    - LLM's reponse to the HumanMessage
    - Can be used for "One-shot" or "Few-shot" learning

`from langchain_cre.messages import HuamMessage, SystemMessage, AIMessage`



## Prompt Templates
- input parameters to the model

### String prompt templates
- to format a single string
- for simple inputs

In [3]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Tell me one {adjective} joke about {topic}"
    )

input_ = {
    "adjective": "funny",
    "topic" : "cats"
}

prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

### Chat prompt templates
- designed to work with chat models
- can assign various roles to the messates
    - system
    - human
    - ai

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about{topic}")
])

input_ = {"topic" : "cates"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke aboutcates', additional_kwargs={}, response_metadata={})])


### MessagePlaceholder
- Special tool for ChatPromptTemplate
- dynamic container for a list of messages
- standard place holder like {topic} expect a single string
- MessagePlaceholder excepts an array of message objects like HumanMessage or AIMessage

#### purpose
- can inject entire history of messages into a template
- code below

|Feature|{variable_name} (String)|MessagesPlaceholder|
|---|---|---|
|Expected Data | A single string. | A list of Message objects.|
|Result | Replaces text inside a message. | Adds multiple messages to the list.|
|Best For,"Keywords | topics, names." | "Chat history, Agent scratches, memory."|

In [5]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    # This is where the magic happens:
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}"),
])

# When you invoke this, you pass a LIST of messages for "chat_history"
# and a STRING for "input".

ModuleNotFoundError: No module named 'langchain.prompts'

## Output parsers
- conver the output of the LLM to a more suitable form
    - like to CSV or Json

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

In [11]:
# JSON Parser
from langchain_core.output_parsers import JsonOutputParser

# Need ot import a BaseModel and Field form langchain to generate the model output
from pydantic import BaseModel, Field

class Joke(BaseModel):
    setup: str = Field(description = "Question to setup a joke")
    punchline: str = Field(description = "Answer to the joke")


joke_query = "Tell me a joke"

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    input_variables = ["joke_query"],
    template = "Generate a joke in JSON format with the following fields: {format_instructions}\n\n{joke_query}",
    partial_variables = {"format_instructions": format_instructions}
)

chain = prompt | llm | output_parser

chain.invoke({"joke_query": joke_query})


{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

- need format instructions
- says LLMs to produce output in a pariticular way that is understandable to the output parser
- in PromptTemplate
    - partial_variables = {"format_insturctions": output_parser.get_fromat_instructions()}

- class passed to the output parser
    - define the schema

    ``` python
    class Movie(BaseModel):
            "name":str = Field(description="the name of the movie")
            # define a field 'name' in output with datatype as str
            "released_year":int = Field(description="the year at which the movie has released")
    ```

- this class has to be passed to the outputpaser so that it came to know about the output of the llm output
- now we need to force the llm to generate output accourding to this schema
    - done through the prompt

```python
    prompt = ChatPrompt(
        template = "Answer the user query.\n{format_instructions}\n{query}\n"
        , input_varibales=['query']
        , partial_variables={"format_instructions":outputparser.get_format_instrutions()}
    )

```


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

class Movie(BaseModel):
    title:str = Field(description="movie title")
    director:str = Field(description="movie director")
    year:int = Field(description="movie released year")
    genre:str = Field(description="movie genre")

json_parser = JsonOutputParser(pydantic_object=Movie)

prompt_template = PromptTemplate(
    template = "Generate information about the movie.\n{format_instructions}\n{movie_name}\n"
    , input_varibales=['movie_name']
    , partial_variables={"format_instructions":json_parser.get_format_instructions()}
)

movie_chain = prompt_template | llm | json_parser

# test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

print(result)



{'title': 'The Matrix', 'director': 'The Wachowskis', 'year': 1999, 'genre': 'Science Fiction'}


## Documents
- Document object contain infroamtion about some data
- 2 attributes
    - page_content:str
    - menta_data:dict

```python
from langchain_core.documents import Document

Document(page_content="""
Python is an interpretted high-level genenral purpose programming language.
"""
metadata={
    "document_id": 2020
    , "docuemnt_source": "About_python"
    , "created_time": 1680013019
}
)
```

## Document loaders
- to load document from a variety of sources
    - eg pdf
- Langchain offers 100 distinct sources
- integration with other major providers
    - AirByte, Unstructured, Amazon S3 buckets

### PDF loader
- load pdf documents

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader

# loader = PyPDFLoader(
#     "./Documents/LangChain Basics.pdf"
#     , extract_images=False
#     )
loader = PyMuPDFLoader(
    "./Documents/LangChain Basics.pdf"
    , extract_images=False
    )

document = loader.load()

# to look at a page
document[2]

Document(metadata={'producer': 'Skia/PDF m143', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36', 'creationdate': '2026-01-16T09:06:50+00:00', 'source': './Documents/LangChain Basics.pdf', 'file_path': './Documents/LangChain Basics.pdf', 'total_pages': 56, 'format': 'PDF 1.4', 'title': 'Skills Network Labs', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-16T09:06:50+00:00', 'trapped': '', 'modDate': "D:20260116090650+00'00'", 'creationDate': "D:20260116090650+00'00'", 'page': 2}, page_content='Setup\nFor this lab, you will use the following libraries:\nibm-watson-ai , ibm-watson-machine-learning  for using LLMs from\nIBM\'s watsonx.ai.\nlangchain , langchain-ibm , langchain-community , langchain-\nexperimental  for using relevant features from LangChain.\npypdf  is an open-source pure-python PDF library capable of splitting, merging,\ncropping, and transforming the pages of PDF files.\nchromadb  is an o

In [3]:
document[1].page_content[:1000]

"In this lab, you will gain hands-on experience using LangChain to simplify the complex\nprocesses required to integrate advanced AI capabilities into practical applications. You\nwill apply core LangChain framework capabilities and use Langchain's innovative\nfeatures to build more intelligent, responsive, and efficient applications.\nTable of contents\n1. Objectives\n2. Setup\nA. Installing required libraries\nB. Importing required libraries\n3. LangChain concepts\nA. Model\nB. Chat model\nC. Chat message\na. Exercise 1: Compare Model Responses with Different Parameters\nD. Prompt templates\nE. Output parsers\na. Exercise 2: Creating and Using a JSON Output Parser\nF. Documents\na. Exercise 3: Working with Document Loaders and Text Splitters\nb. Exercise 4: Building a Simple Retrieval System with LangChain\nG. Memory\na. Exercise 5: Building a Chatbot with Memory using LangChain\nH. Chains\na. Exercise 6: Implementing Multi-Step Processing with Different Chain\nApproaches\nI. Tools a

## Text Splitter
- split the documents to manageable chunks. to fit to models context window
- working
    - split the text into meaningful chunks (often sentences)
    - combine the above chunks to certain size (as measured by a specific function)
    - then start creating a next chunk, with some overlap, to keep the context between the chunks

### Some text splitters
- CharacterTextSplitter
    - split by characters. measures chunk lentgh by number of characters

In [2]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

chunks = text_splitter.split_documents(document)

print(len(chunks))

713


In [4]:
# to take a look at the page content
chunks[5].page_content

'interactions with LLMs. Data scientists can dynamically compare prompts and switch\nbetween foundation models without significant code modifications. These capabilities'

## Embedding models
- generate vector representation of chunks
- semantic search can be performed on this vectors
- one can specify the purpose embeddings

### Task types
|Task Type|Usage|
|---|---|
|RETRIEVAL_QUERY|Optimized for short user question|
|RETRIEVAL_DOCUMENT|Optimized for large blocks of searchable text|
|SEMANTIC_SIMILARITY|Best for comparing if two strings mean the same thing|


In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_api = os.getenv("GEMINI")

embedding_model = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004'
    , task_type="retrieval_document"
    , google_api_key=gemini_api
)

texts = [text.page_content for text in chunks]

embedding_result = embedding_model.embed_documents(texts)
embedding_result[0]

[0.00665653,
 -0.026778936,
 -0.045609027,
 0.0014161806,
 0.023130806,
 0.008418884,
 0.03373465,
 -0.016791815,
 0.020080348,
 0.042548522,
 -0.005672341,
 0.0140526,
 0.039516717,
 0.0026658091,
 -0.010700434,
 -0.047749564,
 0.023608563,
 0.03677314,
 -0.08153242,
 0.03877114,
 0.024119072,
 -0.028291628,
 -0.050841395,
 -0.05424177,
 0.00849343,
 -0.0005243845,
 0.036947247,
 0.029546307,
 0.00058978715,
 -0.040417172,
 0.012301926,
 0.050000094,
 0.009811688,
 -0.062875785,
 -0.00019976383,
 0.0006244626,
 0.04847164,
 0.019516034,
 0.04745189,
 -0.067448065,
 -0.02482002,
 0.021977141,
 -0.010150155,
 0.0708075,
 0.0040558483,
 -0.005313352,
 -0.03605113,
 0.004861062,
 -0.037316535,
 0.0215464,
 0.075100794,
 0.014173465,
 -0.04672312,
 0.012999744,
 -0.07540319,
 0.0015304333,
 -0.04325932,
 -0.00730391,
 0.013826546,
 0.049046088,
 -0.030122366,
 -0.040012438,
 -0.036402162,
 0.01811324,
 -0.008904702,
 -0.020517109,
 -0.011273851,
 -0.043738786,
 -0.016959246,
 0.020142589,


## Vector store
- to store the embeddings
- Many vector stores
    - Chroma

`from langchain.vectorstores import Chroma`

- to search the data from the Chroma db

```python
docsearch = Chroma.from_documents(chunks, embedding_model)
# perform the embeddings ont he chunks and stroe the data in a vector store
````

- to perform a similarity search and retrieve the docuemnt

```python
query = "what is Langchain?"
doc = docsearch.similarity_search(query)
print(doc[0].page_content)
```


In [3]:
import os

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma

document_path = "./Documents/LangChain Basics.pdf"

#lodading
loader = PyMuPDFLoader(
    document_path
    , extract_images=False
)
document = loader.load()

# splitting
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)

document_store = Chroma.from_documents(
    chunks
    , embedding_model
)

query = "what is langchain?"

docs = document_store.similarity_search(
    query
    , k = 3
)
print(docs[0].page_content)

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.


In [4]:
for doc in docs:
    print(doc.page_content)
    print('-------------------------')

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------


## Retrivers
- interface to return docuemnts using unstructured query
- may not be able to store documents
- can use a vector store as the backbone of a retriever
    - other type of retrivers are also exist

In [5]:
retriever = document_store.as_retriever(
    search_kwargs={"k":3} # pass the number of results
)
docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)
    print('-------------------------')

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------


## Parent document retrievers
- you have conflicting goals when you retrieve documents
    - small documents has embeddings that reflects the exact meaning of the content (precision)
    - the document has to be long enough to retain the context of the text (context)
- `ParentDocumentRetriever` strikes the balance
- during retreival, first featches the small chunks, however, look up the Parent IDs to retrieve the large document
- Stores data in 2 diff places
    - vector store (child chunks)
        - 100 - 200 words
        - used for actual search
    - Document Store (Parent Document)
        - entire original document
        - Stored in simple databases (like dictionary or InMemoryStore)
        - Linked to the children
        - Passes the Parent Document to the LLM

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.storage import InMemoryStore
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma

document_path = "./Documents/LangChain Basics.pdf"

#lodading
loader = PyMuPDFLoader(
    document_path
    , extract_images=False
)
document = loader.load()

parent_splitter = CharacterTextSplitter(
    chunk_size=2000
    , chunk_overlap=200
    , separator="\n"
)
child_splitter = CharacterTextSplitter(
    chunk_size=200
    , chunk_overlap=20
    , separator="\n"
)
vectorstore = Chroma(
    collection_name="split_parents"
    , embedding_function=embedding_model
)
# large memory store
store = InMemoryStore()

retriver = ParentDocumentRetriever(
    vectorstore=vectorstore
    , docstore=store
    , child_splitter=child_splitter
    , parent_splitter=parent_splitter
)

# add documents to the hierarchical retrieval system
retriver.add_documents(document)

print(f"no of docs: {len(document)}")

sub_docs = vectorstore.similarity_search("what is langchain?")

print("\n Sub Documents: \n")
for doc in sub_docs:
    print(doc.page_content)
    print('-------------------------')

print("\n Retrieved Documents: \n")
retrived_docs = retriver.invoke("what is langchain?")
print(retrived_docs[0].page_content)

no of docs: 56

 Sub Documents: 

IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for using relevant features from LangChain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------
LangChain provides a lot of utilities for adding memory to a system. These 
utilities can be used by themselves or 
incorporated seamlessly into a chain.
-------------------------
In this example, you will load the following paper about using LangChain. You can access
and read the paper here: Revolutionizing Mental Health Care through LangChain: A
-------------------------

 Retrieved Documents: 

Setup
For this lab, you will use the following libraries:
ibm-watson-ai , ibm-watson-machine-learning  for using LLMs from
IBM's watsonx.ai.
langchain , langchain-ibm , langchain-community , langchain-
experimental  for us

## RetrievalQA
- depricated in 2026
- to retrieve information from a document

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm
    , retriever=retriever
    , chain_type="stuff"
    # how the retrieved documents are combined
    # stuff - all documents are stuffed into the prompt 
    , return_source_documents=False 
    # only return the generated answer
)

query = "Explain the concept of 'Agents' in LangChain."

qa.invoke(query)

{'query': "Explain the concept of 'Agents' in LangChain.",
 'result': "In LangChain, an 'Agent' refers to a system that utilizes a large language model (LLM) as a reasoning engine. The primary function of an Agent is to identify suitable actions and determine the best course of action. \n\nLanguage models, on their own, are limited to generating text and cannot perform actions. Agents, however, leverage these language models to make decisions and take actions, effectively bridging the gap between text output and real-world actions.\n\nIn LangChain, Agents can be created using the `create_react_agent` function and `AgentExecutor`, which work together to enable the Agent to interact with its environment and take actions based on the output of the language model."}

## Memory
- Most LLMs has a conversational inerface
- every conversation being able to refer to information introduced in earlier conversation
- bare minimum, a conversation system sould be able to access some past information

- LLMs are stateless
    - doesn't remember the previous converations
- `ChatHistoryClass is a light-weight storage that sotres 
    - Humen, AI and System messages

### Chat message history
- Core modules 
    - `ChatMessageHistory`
    - `HumanMessages`
    - `AIMessages`

In [21]:
from langchain.memory import ChatMessageHistory

chat = llm
history = ChatMessageHistory()
history.add_ai_message("hi")
history.add_user_message("what is the capital of France?")

In [22]:
history.messages

[AIMessage(content='hi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the capital of France?', additional_kwargs={}, response_metadata={})]

In [23]:
ai_response = chat.invoke(history.messages)
ai_response

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 48, 'total_tokens': 56, 'completion_time': 0.011999896, 'completion_tokens_details': None, 'prompt_time': 0.001521533, 'prompt_tokens_details': None, 'queue_time': 0.048874447, 'total_time': 0.013521429}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019c0aa5-ec58-7502-98db-9b88357585a5-0', usage_metadata={'input_tokens': 48, 'output_tokens': 8, 'total_tokens': 56})

## Conversation buffer
- converation buffer memory allows for storage of messages


In [28]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

converation = ConversationChain(
    llm=llm,
    verbose=False,
    memory=ConversationBufferMemory()   
)

In [29]:
questions = [
    "Hi there!",
    "Can you tell me a joke?",
    "What's the capital of Germany?",
    "Explain the concept of 'Agents' in LangChain."
]  

for question in questions:
    print(f"User: {question}")
    response = converation.invoke(question)
    print(response)

User: Hi there!
{'input': 'Hi there!', 'history': '', 'response': "Hello. It's lovely to chat with you. I've been trained on a vast amount of text data, including but not limited to, books, articles, and conversations, which allows me to generate human-like responses. My training data is based on a snapshot of the internet from 2021, so I might not be aware of very recent events or developments. I'm excited to see where our conversation takes us. Is there something specific you'd like to talk about, or would you like me to suggest some topics? By the way, I can provide information on a wide range of subjects, from science and history to entertainment and culture. For instance, I can tell you about the latest discoveries in the field of astronomy, or discuss the plot of a particular book or movie. What sounds interesting to you?"}
User: Can you tell me a joke?
{'input': 'Can you tell me a joke?', 'history': "Human: Hi there!\nAI: Hello. It's lovely to chat with you. I've been trained on

## Chains
- allows to combine multiple components into cohesive workflows
    - SequantialChain, ParallelChain
    - LCEL
### Purpose of chains
- originally designed to handle single input and generate single response
- real-world applicatons requires
    - multistep reasoning
    - accessing different tools
    - breaking complex tasks to managebale chunks
- chains allow complex workflows

## Evaluation of chain patterns
- Traditional chains
    - LLMChain
    - SequentialChain
- LCEL(| operator)
    - more flexible and functional approach
    - easier to compose and debug
- LCEL is the recommented pattern for new development
- has superior flexibility and expressiveness

In [2]:
# Traditional approch
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

template = """
Your task is to come up with the most traditional dish from the location
given below.
{location}
YOUR ANSWER:
"""

prompt = PromptTemplate(
    input_variables=["location"],
    template=template
)

chain = LLMChain(
    llm=llm, 
    prompt=prompt
)

chain.invoke({"location": "Frankfurt, Germany"})


/tmp/ipykernel_6013/39134590.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'location': 'Frankfurt, Germany',
 'text': 'Frankfurter sausages, also known as Frankfurters or simply "Currywurst" when served with a tomato-based sauce and curry powder, are a classic dish from Frankfurt, Germany. However, a more traditional and iconic dish from Frankfurt is the "Frankfurter Rippchen" or "Rippchen mit Sauerkraut", which consists of cured pork ribs served with sauerkraut and potatoes.\n\nBut the most traditional dish from Frankfurt is probably the "Frankfurter Schnitzel" or more specifically, the "Schnitzel von der Rippchen" is not as common, a dish that is more commonly associated with Frankfurt is the "Green Sauce" (Grüne Soße) which is served with various meats, but the most traditional dish is the "Frankfurter Saumagen" or "Larded Belly" but the dish that is most iconic and widely recognized is the "Frankfurter Rippchen" variant, the "Sauerbraten-style Rippchen" is not as common, the most iconic and widely recognized traditional dish from Frankfurt is the "Frankf

In [3]:
# mordern approch with LCEL
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """
Your task is to come up with the most traditional dish from the location
given below.
{location}
YOUR ANSWER:
"""

prompt = PromptTemplate(
    input_variables=["location"],
    template=template
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"location": "Frankfurt, Germany"})

print(result)

Frankfurter sausages, also known as Frankfurters or simply "Currywurst" when served with a tomato-based sauce and curry powder, are a classic dish from Frankfurt, Germany. However, a more traditional and iconic dish from Frankfurt is the "Frankfurter Rippchen" or "Rippchen mit Kraut", which consists of cured pork ribs served with sauerkraut and potatoes.

But the most traditional dish from Frankfurt is probably the "Frankfurter Schnitzel" or more specifically, the "Schnitzel von der Rippchen" is not as common, a more common dish is the "Sauerbraten" or "Frankfurter Sausage with Sauerkraut and Potatoes" but the most iconic and traditional dish is the "Frankfurter Grüne Soße" (Green Sauce) with meat or fish, but the most famous and iconic is the "Frankfurter Grüne Soße" with Frankfurter Sausages.

However, the most traditional and iconic dish from Frankfurt, Germany, is the "Frankfurter Grüne Soße" with Frankfurter Sausages or the "Frankfurter Sausage with Sauerkraut and Potatoes" but th

In [6]:
location = "malappuram, Kerala"

result = chain.invoke({"location": location})
print(result)

The most traditional dish from Malappuram, Kerala is likely to be "Thalassery Biriyani" or more specifically, "Malabar Biriyani". However, a dish that is more unique to the region is "Kozhikodan Erachi Olathiyathu" or "Malappuram Style Beef Olathiyathu", but the most traditional and widely popular dish is "Ari Pathiri with Chicken or Beef Curry" or simply "Pathiri". 

Pathiri is a traditional Malabar dish, which is a type of flatbread made from rice flour, and is usually served with a non-vegetarian curry, such as chicken or beef. It's a staple food in the region, especially during special occasions and festivals.


## Sequential Chain
- output of one LLM as input for another

In [7]:
# LCEL approach
from pprint import pprint

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# define templates
location_template = """
Your job is to come up with a classic dish from {location}.
YOUR RESPONSE:
"""

dish_template = """
Given a meal {meal}, give me the authentic recipe for it.
YOUR RESPONSE:
"""

time_template = """
Provide an estimated time to prepare the dish {recipe}.
YOUR RESPONSE:
"""

# chains
location_chain = (
    PromptTemplate.from_template(
        template=location_template
    )
    | llm
    | StrOutputParser()
)

dish_chain = (
    PromptTemplate.from_template(
        template=dish_template
    )
    | llm
    | StrOutputParser()
)

time_chain = (
    PromptTemplate.from_template(
        template=time_template
    )
    | llm
    | StrOutputParser()
)

# overall chain
overall_chain = (
    RunnablePassthrough.assign(meal=lambda x: location_chain.invoke(x))
    | RunnablePassthrough.assign(recipie=lambda x: dish_chain.invoke(x))
    | RunnablePassthrough.assign(time=lambda x: time_chain.invoke(x))
)

# run the overall chain
result = overall_chain.invoke({"location": "malappuram, Kerala"})
pprint(result)

{'location': 'malappuram, Kerala',
 'meal': 'Malappuram, a district in Kerala known for its rich culinary '
         'heritage. One classic dish that originates from this region is the '
         '"Thalassery Biriyani" or "Malabar Biriyani". However, I\'d like to '
         'introduce you to a lesser-known but equally delicious dish, the '
         '"Kadala Curry" or more specifically, the "Malappuram-style Kadala '
         'Curry".\n'
         '\n'
         'Kadala Curry is a traditional curry made with chickpeas (kadala) in '
         'a flavorful and aromatic coconut-based gravy. The Malappuram-style '
         'Kadala Curry is unique in that it uses a blend of spices, including '
         'coriander, cumin, fennel, and cinnamon, which are roasted and ground '
         'into a fine paste. This paste is then sautéed with onions, ginger, '
         'and garlic, and cooked with chickpeas, coconut milk, and a touch of '
         'tamarind.\n'
         '\n'
         'The curry is typica

### Note:

- `RunnablePassthrough.assign(recipie=lambda x: dish_chain.invoke(x))` 
- the name recipie is assigned to the second prompt template variable
-```dish_template = """
Given a meal {meal}, give me the authentic recipe for it.
YOUR RESPONSE:
"""```
- this is how the output from first chain is passed to the next chain


In [1]:
from langchain_core.tools import Tool
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL

In [2]:
python_repl = PythonREPL()

python_calculator = Tool(
    name="Python Calculator",
    func=python_repl.run,
    # python_repl.run takes a string input and executes it as python code
    description="A Python shell. Use this to execute python commands and run python code snippets."
)

In [4]:
python_calculator.invoke("a=3;b=4;print(a+b)")

'7\n'

`@tool` decorator

In [ ]:
@tool
def python_calculator_tool(code: str) -> str:
    """A Python shell. Use this to execute python commands and run python code snippets."""
    return python_repl.run(code)

In [5]:
python_calculator.invoke("2 + 2")

''

## Tool kits
- colletion of tools
- designed to be used togher with a specific task
- example
    - `tools = [python_calculator, search_weather]`

## Agents
- language models can't take actions
- they just output text

- An agent system uses the reasoning skills of LLMs to identify the appropriate action, and the input needed for the action
- The results are fed back to the agent
- The agent determines, more actions needed or the task is complete

- LangChain tools
    - create_react_agent
    - AgentExecutor

- agentic workflow
    - ReAct: Reasoning + Act
    - LLMs great at
        - Reasoning howeve prone to hallucination
        - using tools but prone to losing track of the goal
    - ReAct solve these problems by forcing AI to do both in a structural loop

    ### ReAct Loop
    - insted of going to final answer stright, it folows a repetative cycle of 4 distinct steps
        - <b>Thought</b>: The agent writes down what it thinks about the user's request (I need to find the current price of Bitcoin to answer this.)
        - <b>Action</b>: Uses a specific tool (google search the price)
        - <b>Observation</b>: reads the result of the action (Bitcoin current price is $95,000)
        - <b>Repeat/Final answer</b>: The agent looks at the observation and decides if it knows enough to finish. if not, it starts a new thought

## Tools and Agents
### Tools
- Tools provide extra functionality to LLMs
    - search tools: Connect to serch engines, database queries, or vector stores
    - API tools: Make calls to eerternal web services
    - Human-in-the-loop tools: Request human input for critical decisions
- Tools that LangChain supports
    - https://docs.langchain.com/oss/python/langchain/overview#tools

- `Python REPL` tool as an example
    - this runs a python command
    - LLM to generate code to calculate the 
    
- Tools can be incorporated in 2 ways
    - `@tool` decorator
        - `from langchain.tools import tool`
    - Class
        - `from langchain_core.tools impot Tool`
    

In [2]:
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool

In [3]:
prompt_template = PromptTemplate.from_template("""
You are a helpful assistant that can use tools to answer questions.
{tools}
                                               
availabe tools are {tool_names}
                                               
To use tools use the following format:
'''
Thought: I need to figure out what to do
Action: tool_name
Action Input: the input to the tool
'''

After you use a tool, he obeservation will be provided to you:
'''
Observation: the result of the tool
'''

Then you should continue with teh thought-action-observation cycle until you have the final answer. 
When you have the final answer, respond in this format:
                                               
'''
Thought: I know the answer
Final Answer: the final answer to the original query
'''

Remember, when using the Python Calculator tool, the inputs mus be valid

Begin!

Question: {input}
{agent_scratchpad}                                                                                                       
""")

In [ ]:
agnet = create_react_agent(
    llm = llm,
    tools = tools,
    prompt = prompt_template
)